# 08 · 端到端實戰：做一個圖像生成小工具

整條軌道的收尾,也是整個 AI/ML 教學弧線的最後一哩。把學到的東西包成一個**好用的小工具**:輸入一句 prompt,輸出一張圖,還能批次生成、挑最好的。最後聊怎麼把它**分享上線**。

## 1. 安裝與載入

In [ ]:
%pip install -q -U diffusers transformers accelerate

In [ ]:
import torch
from diffusers import AutoPipelineForText2Image

pipe = AutoPipelineForText2Image.from_pretrained(
    "stabilityai/sd-turbo", torch_dtype=torch.float16
).to("cuda")
print("生成器就緒。")

## 2. 把生成包成一個函式

好工具從好介面開始:一個 `generate(prompt, n)` 函式,把細節藏起來。

In [ ]:
import matplotlib.pyplot as plt


def generate(prompt, n=4):
    """給一句 prompt,生 n 張圖並排顯示。"""
    imgs = [pipe(prompt, num_inference_steps=1, guidance_scale=0.0).images[0] for _ in range(n)]
    fig, axes = plt.subplots(1, n, figsize=(3 * n, 3))
    for ax, im in zip(axes, imgs):
        ax.imshow(im); ax.axis("off")
    plt.suptitle(prompt, fontsize=11); plt.tight_layout(); plt.show()
    return imgs


_ = generate("a steaming cup of coffee on a wooden desk, cozy morning light", n=4)

## 3. 批次比較不同 prompt

同一個工具,快速試不同點子——這就是創作流程的雛形。

In [ ]:
for p in ["a robot reading a book in a library, cinematic",
          "a watercolor painting of cherry blossoms"]:
    generate(p, n=4)

## 4. 從 Colab 到一個真的網頁工具

要把它變成別人能用的工具,業界最快的路是 **Gradio + Hugging Face Spaces**:

```python
# import gradio as gr
# def fn(prompt):
#     return pipe(prompt, num_inference_steps=1, guidance_scale=0.0).images[0]
# gr.Interface(fn, inputs="text", outputs="image").launch()
```

- **Gradio** 幾行就生出一個有輸入框 + 圖片輸出的網頁介面(在 Colab 直接 `launch()` 就有公開連結)。
- **Hugging Face Spaces** 可以免費託管這個 app,變成一個任何人都能用的永久網址。
- 這跟 `cv` / `rl` 軌道的「Colab 訓練 → 匯出 → 上線」是同一個精神:**把模型變成別人能用的東西**。

## 軌道小結

你從**加噪/去噪的原理**,一路做到能用、能控制、能上線:

- **生成世界觀**(01)→ **手刻 forward diffusion**(02)→ **手刻去噪 U-Net**(03)→ **DDPM/DDIM 取樣**(04)
- **文字條件 CLIP**(05)→ **diffusers 跑 SD**(06)→ **img2img/inpainting/LoRA 控制**(07)→ **生成工具 + 上線**(08)

**原理你手刻過、工具你也會用、還能做成產品**——這就是生成式 AI 從理解到落地的完整路徑。🎨

## 整個程式實驗室 AI/ML 弧線到此完成

經典 ML → 深度學習 → 從零 LLM → AI Agent → 強化學習 → 電腦視覺 → 資料科學 → 生成式影像。恭喜你走完整條路。🎉